# Per-category `min_ade` — KD arms on LCDrive val (n = 1000)

Every arm is scored **through the teacher's frozen action expert** (`StitchedAlpamayoR1`),
which is the head these arms were trained to drive. Scoring a student's own token head
instead gives a different — and for the CE-free arms, degenerate (`min_ade` 37.5239,
every generation malformed) — answer.

`min_ade` is **best-of-6** samples against ground truth, so it is an *oracle* over modes.
Lower is better. Comparisons are **paired per clip**, never a difference of aggregates.

Colour encodes magnitude **within each row**, so each category is judged on its own scale:
a category where every arm does badly does not paint the whole row dark.

In [1]:
import json, csv, numpy as np, pandas as pd
from pathlib import Path

TRAIN = Path("/data/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training")
SCENARIOS = Path("/data/datasets/physical_ai_av/lcdrive_physicalai_av_manifests"
                 "/lcdrive_val_primary_scenario_mysubset.csv")

# label -> stitched per-clip results file. Ordered ceiling -> floor.
ARMS = {
    "teacher":    "stitch_4b_teacher.json",
    "blockonly":  "stitch_4b_blockonly_checkpoint-1598.json",
    "kvonly_e3":  "stitch_4b_kvonly_e3_checkpoint-4794.json",
    "kvband_e1":  "stitch_4b_kvband_checkpoint-1598.json",
    "kvonly_e1":  "stitch_4b_kvonly.json",
    "ce":         "stitch_4b_ce.json",
}
ARMS = {k: v for k, v in ARMS.items() if (TRAIN / v).exists()}
print("arms found:", ", ".join(ARMS))

arms found: teacher, blockonly, kvonly_e3, kvband_e1, kvonly_e1, ce


In [2]:
# Each stitched JSON is a flat list of per-clip records carrying `clip_id`, which is what
# makes the join to scenario labels -- and the paired tests -- possible at all.
scores = {
    arm: {r["clip_id"]: r["min_ade"] for r in json.loads((TRAIN / f).read_text())}
    for arm, f in ARMS.items()
}
category = {r["clip_uuid"]: r["scenario_category"]
            for r in csv.DictReader(SCENARIOS.open())}

# Restrict to clips every arm scored, so all columns describe the SAME clip set.
clips = [c for c in category if all(c in s for s in scores.values())]
df = pd.DataFrame({arm: [scores[arm][c] for c in clips] for arm in ARMS},
                  index=pd.Index([category[c] for c in clips], name="category"))
print(f"{len(clips)} clips x {len(ARMS)} arms")

table = df.groupby("category").mean()
table.insert(0, "n", df.groupby("category").size())
table = table.sort_values("n", ascending=False)

overall = df.mean().to_frame().T
overall.insert(0, "n", len(clips))
overall.index = ["ALL"]
table = pd.concat([table, overall])

print(table.round(3).to_string())   # plain-text fallback
table.round(3)

1000 clips x 6 arms
                                   n  teacher  blockonly  kvonly_e3  kvband_e1  kvonly_e1      ce
General Training/Validation      341    0.385      1.511      1.670      2.644      1.871   8.957
Lane Keeping Curve                59    0.976      4.300      5.773      6.032      6.574   8.380
Lead Vehicle Following            56    0.625      2.001      2.483      3.446      2.567   9.384
Nudge Static Obstacle Maneuver    56    0.482      1.669      1.725      2.314      1.911   4.529
Lane Keeping                      53    0.554      1.837      2.086      2.761      2.343   4.030
Nudge Maneuver                    53    0.602      1.775      1.812      2.207      1.974   2.963
Speed Control                     53    0.597      3.297      3.160      4.171      3.302   6.575
Stop for Vehicle                  48    0.447      1.490      2.822      3.058      2.980   3.120
Vulnerable Road Users (VRU)       47    0.632      1.401      1.330      1.725      1.414   2.337


,n,teacher,blockonly,kvonly_e3,kvband_e1,kvonly_e1,ce
General Training/Validation,341,0.385,1.511,1.670,2.644,1.871,8.957
Lane Keeping Curve,59,0.976,4.300,5.773,6.032,6.574,8.380
Lead Vehicle Following,56,0.625,2.001,2.483,3.446,2.567,9.384
Nudge Static Obstacle Maneuver,56,0.482,1.669,1.725,2.314,1.911,4.529
Lane Keeping,53,0.554,1.837,2.086,2.761,2.343,4.030
Nudge Maneuver,53,0.602,1.775,1.812,2.207,1.974,2.963
Speed Control,53,0.597,3.297,3.160,4.171,3.302,6.575
Stop for Vehicle,48,0.447,1.490,2.822,3.058,2.980,3.120
Vulnerable Road Users (VRU),47,0.632,1.401,1.330,1.725,1.414,2.337
Lane Change,47,0.880,2.840,3.113,4.253,3.416,12.459


In [3]:
from matplotlib.colors import LinearSegmentedColormap

# Sequential = ONE hue, light -> dark (never a rainbow, never red/green: both fail CVD).
# Light = low min_ade = better.
SEQ = LinearSegmentedColormap.from_list("seq", ["#f2f7fb", "#c8dbeb", "#7fa9cd", "#3d6f9e", "#1b3d5c"])

arm_cols = [c for c in table.columns if c != "n"]

styled = (
    table.style
    # axis=1 -> normalise WITHIN each row, so every category is judged on its own scale.
    .background_gradient(cmap=SEQ, subset=arm_cols, axis=1, text_color_threshold=0.45)
    .format({"n": "{:.0f}", **{c: "{:.3f}" for c in arm_cols}})
    .set_caption("min_ade through the teacher's expert — lower is better; "
                 "colour is row-relative (per category)")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"),
                                          ("padding-bottom", "8px"), ("color", "#444")]},
        {"selector": "th", "props": [("font-weight", "600"), ("text-align", "right")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])
)
styled

,n,teacher,blockonly,kvonly_e3,kvband_e1,kvonly_e1,ce
General Training/Validation,341,0.385,1.511,1.670,2.644,1.871,8.957
Lane Keeping Curve,59,0.976,4.300,5.773,6.032,6.574,8.380
Lead Vehicle Following,56,0.625,2.001,2.483,3.446,2.567,9.384
Nudge Static Obstacle Maneuver,56,0.482,1.669,1.725,2.314,1.911,4.529
Lane Keeping,53,0.554,1.837,2.086,2.761,2.343,4.030
Nudge Maneuver,53,0.602,1.775,1.812,2.207,1.974,2.963
Speed Control,53,0.597,3.297,3.160,4.171,3.302,6.575
Stop for Vehicle,48,0.447,1.490,2.822,3.058,2.980,3.120
Vulnerable Road Users (VRU),47,0.632,1.401,1.330,1.725,1.414,2.337
Lane Change,47,0.880,2.840,3.113,4.253,3.416,12.459


## Paired comparison: `blockonly` − `kvonly_e3`, within category

Negative favours `blockonly`. `z` is a paired one-sample statistic on the per-clip
differences, so it accounts for the fact that some categories are simply harder.

Colour here is **diverging** — two hues with a neutral midpoint at exactly 0 — because the
quantity has polarity (which arm wins), not magnitude.

In [4]:
a, b = "blockonly", "kvonly_e3"
rows = []
for cat, g in df.groupby("category"):
    d = (g[a] - g[b]).to_numpy()
    z = d.mean() / (d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 and d.std() > 0 else np.nan
    rows.append({"category": cat, "n": len(d), f"{a}": g[a].mean(),
                 f"{b}": g[b].mean(), "delta": d.mean(), "z": z})
d_all = (df[a] - df[b]).to_numpy()
rows.append({"category": "ALL", "n": len(d_all), f"{a}": df[a].mean(), f"{b}": df[b].mean(),
             "delta": d_all.mean(),
             "z": d_all.mean() / (d_all.std(ddof=1) / np.sqrt(len(d_all)))})
paired = pd.DataFrame(rows).set_index("category")

# Diverging: two hues + a NEUTRAL GREY midpoint, symmetric about 0 so the midpoint is
# genuinely "no difference" rather than wherever the data happens to centre.
DIV = LinearSegmentedColormap.from_list(
    "div", ["#1b3d5c", "#7fa9cd", "#eceff1", "#e0a367", "#8a4b12"])
lim = float(np.nanmax(np.abs(paired["delta"])))

styled_paired = (paired.style
   .background_gradient(cmap=DIV, subset=["delta"], vmin=-lim, vmax=lim, text_color_threshold=0.45)
   .format({"n": "{:.0f}", a: "{:.3f}", b: "{:.3f}", "delta": "{:+.3f}", "z": "{:+.2f}"})
   .set_caption(f"{a} − {b} per category · negative favours {a} · |z| > 2 ≈ significant")
   .set_table_styles([
       {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"),
                                         ("padding-bottom", "8px"), ("color", "#444")]},
       {"selector": "th.row_heading", "props": [("text-align", "left")]},
       {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
   ]))

print(paired.round(3).to_string())   # plain-text fallback
styled_paired

                                   n  blockonly  kvonly_e3  delta      z
category                                                                
Cut-In                            36      1.677      1.606  0.071  0.688
General Training/Validation      341      1.511      1.670 -0.158 -2.051
Intersection Navigation           42      1.593      1.762 -0.169 -0.990
Lane Change                       47      2.840      3.113 -0.272 -1.008
Lane Keeping                      53      1.837      2.086 -0.249 -1.398
Lane Keeping Curve                59      4.300      5.773 -1.473 -3.764
Lead Vehicle Following            56      2.001      2.483 -0.482 -2.704
Merging                           44      2.046      1.892  0.155  1.505
Nudge Maneuver                    53      1.775      1.812 -0.037 -0.329
Nudge Static Obstacle Maneuver    56      1.669      1.725 -0.056 -0.503
Speed Control                     53      3.297      3.160  0.137  1.491
Stop for Vehicle                  48      1.490    

,n,blockonly,kvonly_e3,delta,z
category,,,,,
Cut-In,36,1.677,1.606,+0.071,+0.69
General Training/Validation,341,1.511,1.670,-0.158,-2.05
Intersection Navigation,42,1.593,1.762,-0.169,-0.99
Lane Change,47,2.840,3.113,-0.272,-1.01
Lane Keeping,53,1.837,2.086,-0.249,-1.40
Lane Keeping Curve,59,4.300,5.773,-1.473,-3.76
Lead Vehicle Following,56,2.001,2.483,-0.482,-2.70
Merging,44,2.046,1.892,+0.155,+1.51
Nudge Maneuver,53,1.775,1.812,-0.037,-0.33
